# Clase 4 · 04 · Garantías de entrega, checkpoints e idempotencia

En un sistema distribuido algo siempre falla: un productor reintenta un envío, un job se cae a mitad de camino, alguien borra un checkpoint. La teoría distingue tres garantías:

| Garantía | Qué significa | Riesgo |
|---|---|---|
| **At-most-once** | Cada evento se procesa 0 o 1 vez | Se pueden **perder** eventos |
| **At-least-once** | Cada evento se procesa 1 o más veces | Se pueden **duplicar** resultados |
| **Exactly-once** | El efecto es como si se procesara una sola vez | Requiere checkpoint **+** escritura idempotente |

Vamos a provocar cada situación con los eventos válidos del notebook 02 y contar las filas que terminan en el destino.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_rules

In [ ]:
%run ../common/streaming_lab

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
for table in [customers, products]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá la clase 2."
T = lambda name: qualified_table(config, name)

In [ ]:
silver_table = T("stream_silver_classified")
assert spark.catalog.tableExists(silver_table), "Ejecutá primero 02_ingesta_y_enriquecimiento"
LAB = "garantias"
reset_lab(spark, dbutils, config, LAB, ["stream_sink_naive", "stream_sink_dedup", "stream_sink_merge"])
checkpoint = lambda name: lab_path(config, LAB, "checkpoints", name)

def valid_events():
    return (spark.readStream.table(silver_table).where("quality_reason IS NULL AND event_type = 'purchase'")
            .select("event_id", "event_ts", "customer_id", "amount", "payment_channel", "source_batch_id"))

source = spark.table(silver_table).where("quality_reason IS NULL AND event_type = 'purchase'")
total, distinct = source.count(), source.select("event_id").distinct().count()
print(f"Compras válidas en Silver: {total} filas · {distinct} event_id distintos → {total - distinct} duplicados")
display(source.where(F.col("event_id") == "e002_000").select("event_id", "event_ts", "amount", "source_batch_id"))

`e002` aparece dos veces porque el productor lo reenvió en otra llegada. Es el efecto típico de **at-least-once** en el productor: ante la duda, reintenta.

## 1. Sin deduplicar

In [ ]:
results = []
run_stream(valid_events(), checkpoint("naive"), table=T("stream_sink_naive"))
results.append(("1. Append sin deduplicar", spark.table(T("stream_sink_naive")).count()))
print(results[-1])

## 2. Deduplicar por `event_id` con watermark

`dropDuplicatesWithinWatermark` recuerda los `event_id` vistos **mientras estén dentro del watermark**. No guarda todos los IDs de la historia: el estado se limpia a medida que avanza el tiempo.

In [ ]:
deduplicated = valid_events().withWatermark("event_ts", "10 minutes").dropDuplicatesWithinWatermark(["event_id"])
run_stream(deduplicated, checkpoint("dedup"), table=T("stream_sink_dedup"))
results.append(("2. Deduplicado", spark.table(T("stream_sink_dedup")).count()))
print(results[-1])

## 3. Reejecutar con el mismo checkpoint

Simulamos que el job se reinicia (por ejemplo, después de una caída). El checkpoint sabe hasta dónde procesó: no vuelve a escribir nada.

In [ ]:
run_stream(deduplicated, checkpoint("dedup"), table=T("stream_sink_dedup"))
results.append(("3. Reinicio con el mismo checkpoint", spark.table(T("stream_sink_dedup")).count()))
print(results[-1])

## 4. Perder el checkpoint

Ahora alguien borra el checkpoint (o usa otro por error). La consulta no sabe qué procesó y **empieza de nuevo**. Como el destino es un `append` común, todo se escribe **otra vez**: duplicamos resultados.

In [ ]:
deduplicated_again = valid_events().withWatermark("event_ts", "10 minutes").dropDuplicatesWithinWatermark(["event_id"])
run_stream(deduplicated_again, checkpoint("dedup_perdido"), table=T("stream_sink_dedup"))
results.append(("4. Checkpoint perdido + append", spark.table(T("stream_sink_dedup")).count()))
print(results[-1])

## 5. Escritura idempotente con `MERGE`

La solución de la teoría: que **aplicar dos veces tenga el mismo efecto que aplicar una**. Con `foreachBatch` escribimos cada microbatch con un `MERGE` por `event_id`: si el evento ya está, no se inserta. Repetimos la prueba anterior: dos ejecuciones con checkpoints distintos.

In [ ]:
merge_table = T("stream_sink_merge")
spark.sql(f"""CREATE TABLE {merge_table} (event_id STRING, event_ts TIMESTAMP, customer_id BIGINT,
              amount DECIMAL(12,2), payment_channel STRING, source_batch_id STRING) USING DELTA""")

def merge_batch(batch_df, batch_id):
    batch_df.dropDuplicates(["event_id"]).createOrReplaceTempView("incoming_events")
    batch_df.sparkSession.sql(f"""
        MERGE INTO {merge_table} t USING incoming_events s ON t.event_id = s.event_id
        WHEN NOT MATCHED THEN INSERT *""")

run_stream(valid_events(), checkpoint("merge_a"), foreach_batch=merge_batch)
results.append(("5a. MERGE, primera ejecución", spark.table(merge_table).count()))
run_stream(valid_events(), checkpoint("merge_b_perdido"), foreach_batch=merge_batch)
results.append(("5b. MERGE, checkpoint perdido", spark.table(merge_table).count()))
print(results[-2:])

## 6. Resultado

In [ ]:
summary = pd.DataFrame(results, columns=["escenario", "filas en el destino"])
display(summary)

colors = ["#f59e0b", "#16a34a", "#16a34a", "#dc2626", "#2563eb", "#2563eb"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(summary["escenario"][::-1], summary["filas en el destino"][::-1], color=colors[::-1])
ax.axvline(distinct, color="black", ls="--", label=f"compras únicas reales = {distinct}")
for i, v in enumerate(summary["filas en el destino"][::-1]):
    ax.text(v, i, f" {v}", va="center")
ax.set_xlabel("Filas en el destino")
ax.set_title("Exactly-once = checkpoint + escritura idempotente")
ax.legend(loc="lower right")
plt.show()

## Resumen

- El productor reintenta → hay **duplicados** en la entrada; se resuelven con una clave de evento (`event_id`).
- El **checkpoint** evita reprocesar al reiniciar: es lo que da exactly-once **dentro** de Spark.
- Si el checkpoint se pierde, un destino `append` recibe todo otra vez (comportamiento **at-least-once**).
- Una escritura **idempotente** (`MERGE` por clave, `SET x = 5` en lugar de `INCREMENT x`) hace que reprocesar no cambie el resultado.